In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Shared Plotly theme applied to every chart
PLOTLY_LAYOUT = dict(
    template        = 'plotly_dark',
    paper_bgcolor   = '#1a1a2e',
    plot_bgcolor    = '#1a1a2e',
    font            = dict(family='monospace', color='#ccc'),
    margin          = dict(t=60, b=40, l=40, r=40),
    legend          = dict(bgcolor='rgba(0,0,0,0)', bordercolor='#333', borderwidth=1),
)

# One colour per segment — used in every chart
SEG_COLORS = {
    'Champions':           '#4ecb8d',
    'Loyal Customers':     '#7c6fcd',
    'Potential Loyalists': '#64b5f6',
    'Recent Customers':    '#81d4fa',
    'Promising':           '#a5d6a7',
    'Need Attention':      '#ffb74d',
    'About To Sleep':      '#ff8a65',
    'At Risk':             '#ef5350',
    "Can't Lose Them":     '#e91e63',
    'Lost':                '#9e9e9e',
}


In [ ]:
df = pd.read_csv("/content/customer_summary.csv")

print(f'Loaded {len(df):,} customers')
print(f'Columns: {df.columns.tolist()}')
print(f'Recency range: {df["recency_days"].min()} – {df["recency_days"].max()} days')
print(f'Revenue range: £{df["total_revenue"].min():.0f} – £{df["total_revenue"].max():,.0f}')
df.head()

Loaded 4,338 customers
Columns: ['CustomerID', 'first_purchase', 'last_purchase', 'total_orders', 'total_items', 'total_revenue', 'avg_order_value', 'unique_products', 'unique_countries', 'recency_days', 'tenure_days']
Recency range: 1 – 374 days
Revenue range: £4 – £280,206


,CustomerID,first_purchase,last_purchase,total_orders,total_items,total_revenue,avg_order_value,unique_products,unique_countries,recency_days,tenure_days
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,1,74215,77183.60,77183.600000,1,1,326,326
1,12347,2010-12-07 14:57:00,2011-12-07 15:52:00,7,2458,4310.00,23.681319,103,1,2,367
2,12348,2010-12-16 19:09:00,2011-09-25 13:13:00,4,2341,1797.24,57.975484,22,1,75,358
3,12349,2011-11-21 09:51:00,2011-11-21 09:51:00,1,631,1757.55,24.076027,73,1,19,19
4,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,197,334.40,19.670588,17,1,310,310


In [ ]:
# R_score — REVERSED labels because low recency_days = recent = GOOD
df['R_score'] = pd.qcut(
    df['recency_days'].rank(method='first'),
    q=5, labels=[5, 4, 3, 2, 1]
).astype(int)

# F_score — normal order, more orders = higher score
df['F_score'] = pd.qcut(
    df['total_orders'].rank(method='first'),
    q=5, labels=[1, 2, 3, 4, 5]
).astype(int)

# M_score — normal order, more revenue = higher score
df['M_score'] = pd.qcut(
    df['total_revenue'].rank(method='first'),
    q=5, labels=[1, 2, 3, 4, 5]
).astype(int)

# Combined string e.g. '5-5-5' = best customer
df['RFM_Score'] = (df['R_score'].astype(str) + '-' +
                   df['F_score'].astype(str) + '-' +
                   df['M_score'].astype(str))

# Weighted total score (R matters most)
df['RFM_Total'] = (df['R_score'] * 0.4 +
                   df['F_score'] * 0.3 +
                   df['M_score'] * 0.3).round(2)

print('RFM scores calculated')
print('\nR_score distribution (should be ~868 per bucket):')
print(df['R_score'].value_counts().sort_index().to_string())
print('\nSample of scored customers:')
df[['CustomerID','recency_days','total_orders','total_revenue',
    'R_score','F_score','M_score','RFM_Score','RFM_Total']].head(10)

RFM scores calculated

R_score distribution (should be ~868 per bucket):
R_score
1    868
2    867
3    868
4    867
5    868

Sample of scored customers:


,CustomerID,recency_days,total_orders,total_revenue,R_score,F_score,M_score,RFM_Score,RFM_Total
0,12346,326,1,77183.60,1,1,5,1-1-5,2.2
1,12347,2,7,4310.00,5,5,5,5-5-5,5.0
2,12348,75,4,1797.24,2,4,4,2-4-4,3.2
3,12349,19,1,1757.55,4,1,4,4-1-4,3.1
4,12350,310,1,334.40,1,1,2,1-1-2,1.3
5,12352,36,8,2506.04,3,5,5,3-5-5,4.2
6,12353,204,1,89.00,1,1,1,1-1-1,1.0
7,12354,232,1,1079.40,1,1,4,1-1-4,1.9
8,12355,214,1,459.40,1,1,2,1-1-2,1.3
9,12356,23,3,2811.43,4,3,5,4-3-5,4.0


In [ ]:
def assign_segment(r, f):
    """Return segment name based on R_score and F_score (each 1–5)."""
    if   r >= 5 and f >= 5:  return 'Champions'
    elif r >= 4 and f >= 4:  return 'Loyal Customers'
    elif r >= 5 and f <= 2:  return 'Recent Customers'
    elif r >= 3 and f >= 3:  return 'Potential Loyalists'
    elif r >= 4 and f <= 1:  return 'Promising'
    elif r <= 2 and f >= 3:  return 'At Risk'
    elif r <= 1 and f >= 4:  return "Can't Lose Them"
    elif r <= 2 and f <= 2:  return 'About To Sleep'
    elif r == 3 and f <= 2:  return 'Need Attention'
    else:                    return 'Lost'

df['Segment'] = df.apply(
    lambda row: assign_segment(row['R_score'], row['F_score']),
    axis=1
)

print('\nCustomers per segment:')
print(df['Segment'].value_counts().to_string())


Customers per segment:
Segment
About To Sleep         1074
Potential Loyalists     821
Loyal Customers         682
At Risk                 661
Champions               439
Need Attention          351
Lost                    116
Recent Customers         99
Promising                95


In [ ]:
seg_summary = df.groupby('Segment').agg(
    Customers     = ('CustomerID',    'count'),
    Avg_Recency   = ('recency_days',  'mean'),
    Avg_Frequency = ('total_orders',  'mean'),
    Avg_Revenue   = ('total_revenue', 'mean'),
    Total_Revenue = ('total_revenue', 'sum'),
).round(1).reset_index().sort_values('Customers', ascending=False)

seg_summary['Revenue_Share_%'] = (
    seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100
).round(1)

seg_summary['Color'] = seg_summary['Segment'].map(SEG_COLORS)

print('Segment Summary:')
seg_summary.drop(columns='Color')

Segment Summary:


,Segment,Customers,Avg_Recency,Avg_Frequency,Avg_Revenue,Total_Revenue,Revenue_Share_%
0,About To Sleep,1074,216.7,1.1,488.7,524889.7,5.9
6,Potential Loyalists,821,37.2,3.7,1687.7,1385600.3,15.5
4,Loyal Customers,682,17.7,6.2,2685.3,1831395.9,20.6
1,At Risk,661,150.6,3.4,1250.3,826438.4,9.3
2,Champions,439,5.7,16.0,9204.0,4040565.6,45.3
5,Need Attention,351,52.5,1.2,454.0,159349.5,1.8
3,Lost,116,23.5,1.4,507.8,58902.4,0.7
8,Recent Customers,99,7.1,1.3,503.0,49795.7,0.6
7,Promising,95,23.0,1.0,362.8,34470.5,0.4


## Chart 1: Customers per segment


In [ ]:
seg_bar = seg_summary.sort_values('Customers', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    x            = seg_bar['Customers'],
    y            = seg_bar['Segment'],
    orientation  = 'h',
    marker_color = seg_bar['Color'],
    text         = seg_bar['Customers'].apply(lambda x: f'{x:,}'),
    textposition = 'outside',
    hovertemplate = (
        '<b>%{y}</b><br>'
        'Customers: %{x:,}<br>'
        '<extra></extra>'
    )
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Customer count by segment', font=dict(size=16)),
    xaxis  = dict(title='Number of customers', gridcolor='#2a2a38'),
    yaxis  = dict(title=''),
    height = 450,
    showlegend = False,
)

fig.show()

 ## Chart 2: Revenue share by segment

In [ ]:
fig = go.Figure()

fig.add_trace(go.Pie(
    labels       = seg_summary['Segment'],
    values       = seg_summary['Total_Revenue'],
    hole         = 0.45,
    marker_colors= seg_summary['Color'],
    textinfo     = 'percent+label',
    textfont     = dict(size=11),
    hovertemplate = (
        '<b>%{label}</b><br>'
        'Revenue: £%{value:,.0f}<br>'
        'Share: %{percent}<br>'
        '<extra></extra>'
    )
))

# Centre annotation
total_rev = seg_summary['Total_Revenue'].sum()
fig.add_annotation(
    text       = f'£{total_rev/1000:.0f}K<br>total',
    x=0.5, y=0.5,
    showarrow  = False,
    font       = dict(size=16, color='#eee'),
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Revenue share by segment', font=dict(size=16)),
    height = 500,
)

fig.show()

champ = seg_summary[seg_summary['Segment'] == 'Champions'].iloc[0]
print(f'Champions: {champ["Customers"]:.0f} customers = {champ["Revenue_Share_%"]}% of all revenue')

Champions: 439 customers = 45.3% of all revenue


## Chart 3: Recency vs Frequency scatter
Each dot is one customer.
Dot size = total revenue spent.

In [ ]:
sample = df.sample(min(2000, len(df)), random_state=42).copy()

# Scale dot size between 4 and 20
rev_min, rev_max = sample['total_revenue'].min(), sample['total_revenue'].max()
sample['dot_size'] = 4 + 16 * (sample['total_revenue'] - rev_min) / (rev_max - rev_min + 1)

fig = px.scatter(
    sample,
    x              = 'recency_days',
    y              = 'total_orders',
    color          = 'Segment',
    color_discrete_map = SEG_COLORS,
    size           = 'dot_size',
    size_max       = 18,
    hover_data     = {
        'CustomerID':    True,
        'total_revenue': ':,.0f',
        'RFM_Score':     True,
        'dot_size':      False,
    },
    opacity        = 0.7,
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='Recency vs frequency — each dot is one customer', font=dict(size=16)),
    xaxis  = dict(title='Recency (days since last purchase) — lower is better', gridcolor='#2a2a38'),
    yaxis  = dict(title='Total orders (frequency)', gridcolor='#2a2a38'),
    height = 520,
)

fig.show()

## Chart 4: Average recency and revenue per segment (side-by-side bars)
Two metrics in one chart. Helps compare segment health at a glance.

In [ ]:
seg_sorted = seg_summary.sort_values('Avg_Revenue', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Average recency (days) — lower is better',
                    'Average revenue per customer (£)'),
    horizontal_spacing=0.12
)

# Left: average recency
fig.add_trace(
    go.Bar(
        x            = seg_sorted['Segment'],
        y            = seg_sorted['Avg_Recency'],
        marker_color = seg_sorted['Color'],
        name         = 'Avg recency (days)',
        hovertemplate = '<b>%{x}</b><br>Avg recency: %{y:.0f} days<extra></extra>',
    ),
    row=1, col=1
)

# Right: average revenue
fig.add_trace(
    go.Bar(
        x            = seg_sorted['Segment'],
        y            = seg_sorted['Avg_Revenue'],
        marker_color = seg_sorted['Color'],
        name         = 'Avg revenue (£)',
        hovertemplate = '<b>%{x}</b><br>Avg revenue: £%{y:,.0f}<extra></extra>',
    ),
    row=1, col=2
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title      = dict(text='Segment health — recency vs revenue', font=dict(size=16)),
    height     = 420,
    showlegend = False,
)
fig.update_xaxes(tickangle=35, tickfont=dict(size=9))
fig.update_yaxes(gridcolor='#2a2a38')

fig.show()

## Chart 5: RFM score heatmap
Shows how many customers sit at each R vs F score combination.
Darker = more customers at that combination.

In [ ]:
heatmap_data = (
    df.groupby(['R_score', 'F_score'])
    .size()
    .reset_index(name='count')
    .pivot(index='R_score', columns='F_score', values='count')
    .fillna(0)
    .sort_index(ascending=False)  # R=5 at top
)

fig = go.Figure(go.Heatmap(
    z             = heatmap_data.values,
    x             = [f'F={c}' for c in heatmap_data.columns],
    y             = [f'R={r}' for r in heatmap_data.index],
    colorscale    = 'Purples',
    text          = heatmap_data.values.astype(int),
    texttemplate  = '%{text}',
    textfont      = dict(size=12),
    hovertemplate = 'R_score: %{y}<br>F_score: %{x}<br>Customers: %{z}<extra></extra>',
    showscale     = True,
    colorbar      = dict(title='Customers'),
))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title  = dict(text='RFM heatmap — how many customers at each R × F combination', font=dict(size=16)),
    xaxis  = dict(title='Frequency score (F) — higher is better'),
    yaxis  = dict(title='Recency score (R) — higher is better'),
    height = 420,
)

fig.show()
print('Top-right cell (R=5, F=5) = Champions. Bottom-left (R=1, F=1) = Lost.')

Top-right cell (R=5, F=5) = Champions. Bottom-left (R=1, F=1) = Lost.


## Chart 6: RFM Total score distribution per segment (box plot)
Shows the spread of composite RFM scores inside each segment.


In [ ]:
# Order segments from highest to lowest median RFM_Total
seg_order = (
    df.groupby('Segment')['RFM_Total']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = go.Figure()

for seg in seg_order:
    seg_data = df[df['Segment'] == seg]['RFM_Total']
    fig.add_trace(go.Box(
        y             = seg_data,
        name          = seg,
        marker_color  = SEG_COLORS.get(seg, '#888'),
        boxmean       = True,   # show mean as dashed line
        hovertemplate = (
            f'<b>{seg}</b><br>'
            'RFM total: %{y:.2f}<br>'
            '<extra></extra>'
        )
    ))

fig.update_layout(
    **PLOTLY_LAYOUT,
    title      = dict(text='RFM total score distribution per segment', font=dict(size=16)),
    yaxis      = dict(title='RFM total score (max = 5.0)', gridcolor='#2a2a38'),
    xaxis      = dict(tickangle=25, tickfont=dict(size=9)),
    height     = 460,
    showlegend = False,
)

fig.show()
print('Dashed line inside each box = mean. Solid line = median.')

Dashed line inside each box = mean. Solid line = median.


## Chart 7: Treemap
segments by customer count and revenue
Box size = number of customers.

Box colour = average revenue per customer.


In [ ]:
fig = px.treemap(
    seg_summary,
    path        = ['Segment'],
    values      = 'Customers',
    color       = 'Avg_Revenue',
    color_continuous_scale = 'Teal',
    custom_data = ['Total_Revenue', 'Revenue_Share_%', 'Avg_Recency'],
)

fig.update_traces(
    hovertemplate = (
        '<b>%{label}</b><br>'
        'Customers: %{value:,}<br>'
        'Total revenue: £%{customdata[0]:,.0f}<br>'
        'Revenue share: %{customdata[1]:.1f}%<br>'
        'Avg recency: %{customdata[2]:.0f} days<br>'
        '<extra></extra>'
    ),
    textfont = dict(size=13),
)

fig.update_layout(
    **PLOTLY_LAYOUT,
    title       = dict(
        text = 'Treemap — box size = customers, colour = avg revenue per customer',
        font = dict(size=15)
    ),
    height      = 480,
    coloraxis_colorbar = dict(title='Avg revenue (£)'),
)

fig.show()